Copyright 2025 Google LLC。

In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

<table align="left"> <td>      <a target="_blank" href="https://colab.research.google.com/github/google-gemma/cookbook/blob/main/.archive/FunctionGemma/%5BFunctionGemma%5DFinetune_FunctionGemma_270M_for_Mobile_Actions_with_Hugging_Face.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
</td>
</table>

# 針對行動操作微調 FunctionGemma 270M

notebook 對 FunctionGemma 進行微調，以完成透過 Hugging Face Transformer 強化學習 ([TRL](https://huggingface.co/docs/trl/en/index)) library 接受使用者要求執行行動操作的任務。
在 Google Colab A100 GPU 加速器上訓練 [FunctionGemma 270M](https://huggingface.co/google/functiongemma-270m-it) 時，此過程可能需要 60 分鐘的端到端時間。執行每個程式碼片段以：
1. 設定Colab環境
2. 為fine-tuning準備一個dataset
3. 載入並測試基礎FunctionGemma 270M 模型
4. 微調模型
5. 測試、評估並保存模型以供進一步使用
6. 將checkpoint 轉換為`.litertlm` 進行部署

## 先決條件

此 Colab 需要 **A100 GPU**。您將需要 Colab Pro 訂閱或 Colab Pay to Go 使用積分。
## 設定開發環境

第一步是使用 `pip` 軟體包安裝程式安裝必要的庫。

In [1]:
%pip install torch
%pip install -U transformers==4.57.1 trl==0.25.1 datasets==4.4.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 127.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 465.5/465.5 kB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 49.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 49.7 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.3
    Uninstalling transformers-4.57.3:
      Successfully uninstalled transformers-4.57.3
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


**您可能需要重新啟動會話 (runtime) 才能使用新安裝的庫。 **

##啟用Hugging Face權限
要使用 Gemma 模型，您需要接受模型使用授權並建立存取權杖：
1. **接受[模型頁](http://huggingface.co/google/functiongemma-270m-it) 上的許可證**。

2. **取得具有「寫入」存取權限的有效[存取權杖](https://huggingface.co/settings/tokens)（非常重要！）**

3. **在左側工具列中建立新的Colab secret**。指定`HF_TOKEN` 作為“名稱”，添加您唯一的token 作為“值”，然後開啟“notebook 訪問”。

In [ ]:
from google.colab import userdata
from huggingface_hub import login

# Login into Hugging Face Hub
hf_token = userdata.get('HF_TOKEN')
login(hf_token)

## 載入模型

您可以透過接受授權條款從Hugging Face Hub存取[FunctionGemma 270M](https://huggingface.co/google/functiongemma-270m-it)。該模型的指令調整版本已經接受瞭如何遵循指示的培訓，並且透過 fine-tuning，您現在將使其適應新任務。

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM

gemma_model = "google/functiongemma-270m-it"
base_model = AutoModelForCausalLM.from_pretrained(
    gemma_model,
    device_map="auto",
    attn_implementation="eager",
    dtype="auto")
tokenizer = AutoTokenizer.from_pretrained(gemma_model)

print(f"Device: {base_model.device}")
print(f"DType:  {base_model.dtype}")

config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/536M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/176 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/63.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/706 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/13.8k [00:00<?, ?B/s]

Device: cuda:0
DType:  torch.bfloat16


如果您使用的是 GPU runtime，裝置應列印為`cuda`。

## 加載dataset

為了微調FunctionGemma，我們利用[移動操作
dataset](https://huggingface.co/datasets/google/mobile-actions)，即
可在 Hugging Face 上公開取得。此 dataset 中的每個條目均提供：
*   模型可以使用的工具集（功能）：
    1. Turn the flashlight on
    2. Turn the flashlight off
    3. Create a contact in the phone's contact list
    4. Send an email
    5. Show a location on the map
    6. Open the WiFi settings
    7. Create a new calendar event
*   system prompt 提供當前日期和時間等上下文
*   使用者prompt，例如`turn on the flashlight`。
*   預期的模型響應，包括適當的函數呼叫。

In [7]:
import json
from random import randint
from datasets import load_dataset
from transformers import AutoTokenizer
from huggingface_hub import hf_hub_download

data_file = hf_hub_download(repo_id="google/mobile-actions", filename="dataset.jsonl", repo_type="dataset")
dataset = load_dataset("text", data_files=data_file, encoding="utf-8")["train"].shuffle()

print(f"\n\033[1mHere's an example from your dataset:\033[0m \n{json.dumps(json.loads(dataset[randint(0, len(dataset) - 1)]['text']), indent=2)}")


Here's an example from your dataset: 
{
  "metadata": "eval",
  "tools": [
    {
      "function": {
        "name": "send_email",
        "description": "Sends an email.",
        "parameters": {
          "type": "OBJECT",
          "properties": {
            "body": {
              "type": "STRING",
              "description": "The body of the email."
            },
            "to": {
              "type": "STRING",
              "description": "The email address of the recipient."
            },
            "subject": {
              "type": "STRING",
              "description": "The subject of the email."
            }
          },
          "required": [
            "to",
            "subject"
          ]
        }
      }
    },
    {
      "function": {
        "name": "turn_on_flashlight",
        "description": "Turns the flashlight on.",
        "parameters": {
          "type": "OBJECT",
          "properties": {}
        }
      }
    },
    {
      "function": {
    

## 處理 dataset 以進行培訓和評估

現在您已載入數據，請將訓練 dataset 格式化為 [提示完成](https://huggingface.co/docs/trl/main/en/dataset_formats#prompt-completion) 格式，以便日後進行更有效的訓練 (`completion_only_loss=True`)。這意味著模型將僅從`completion`而不是`prompt`學習。
- `prompt` 對於不可訓練的部分
- `completion` 用於可訓練部分

In [8]:
import json

def apply_format(sample):
  template_iputs = json.loads(sample['text'])

  prompt_and_completion = tokenizer.apply_chat_template(
    template_iputs['messages'],
    tools=template_iputs['tools'],
    tokenize=False,
    # add_generation_prompt is False since we don't need model output after all
    # messages.
    add_generation_prompt=False)

  prompt = tokenizer.apply_chat_template(
    template_iputs['messages'][:-1],
    tools=template_iputs['tools'],
    tokenize=False,
    # add_generation_prompt is True since we would like to include
    # "<start_of_turn>model" in the prompt, if needed.
    add_generation_prompt=True)

  completion = prompt_and_completion[len(prompt):]

  return {
     "prompt": prompt,
     "completion": completion,
     "split": template_iputs["metadata"],
  }

processed_dataset = dataset.map(apply_format)

Map:   0%|          | 0/9654 [00:00<?, ? examples/s]

In [9]:
#@title Review the processed dataset

print("\033[1mHere's an example from the formatted dataset:\033[0m")
print(json.dumps(processed_dataset[randint(0, len(processed_dataset) - 1)], indent=2))

longest_example = max(processed_dataset, key=lambda example: len(example['prompt'] + example['completion']))
longest_example_token_count = len(tokenizer.tokenize(longest_example['prompt'] + longest_example['completion']))

print(f"\n\033[1mThe longest example length is {len(longest_example['prompt'] + longest_example['completion'])} with {longest_example_token_count} tokens. We need to set the max_length larger than the token count in SFTConfig below.\033[0m")
print(json.dumps(longest_example, indent=2))

max_token_count = longest_example_token_count + 100
print(f"\n\033[1mUsing max_token_count of {max_token_count} (= {longest_example_token_count} + 100) for training below.\033[0m")

Here's an example from the formatted dataset:
{
  "text": "{\"metadata\": \"train\", \"tools\": [{\"function\": {\"name\": \"send_email\", \"description\": \"Sends an email.\", \"parameters\": {\"type\": \"OBJECT\", \"properties\": {\"subject\": {\"type\": \"STRING\", \"description\": \"The subject of the email.\"}, \"body\": {\"type\": \"STRING\", \"description\": \"The body of the email.\"}, \"to\": {\"type\": \"STRING\", \"description\": \"The email address of the recipient.\"}}, \"required\": [\"to\", \"subject\"]}}}, {\"function\": {\"name\": \"show_map\", \"description\": \"Shows a location on the map.\", \"parameters\": {\"type\": \"OBJECT\", \"properties\": {\"query\": {\"type\": \"STRING\", \"description\": \"The location to search for. May be the name of a place, a business, or an address.\"}}, \"required\": [\"query\"]}}}, {\"function\": {\"name\": \"turn_off_flashlight\", \"description\": \"Turns the flashlight off.\", \"parameters\": {\"type\": \"OBJECT\", \"properties\": 

In [10]:
#@title Prepare train and eval dataset.

train_dataset = processed_dataset.filter(lambda example: example['split'] == 'train')
eval_dataset = processed_dataset.filter(lambda example: example['split'] == 'eval')

Filter:   0%|          | 0/9654 [00:00<?, ? examples/s]

Filter:   0%|          | 0/9654 [00:00<?, ? examples/s]

## 推薦：測試基本模型

現在，我們已經加載了基本模型和dataset。我們先檢視一下基礎模型的反應能力如何
隨機樣本。
嘗試測試幾次。

In [ ]:
#@title Test with a prompt

from transformers import pipeline
from random import randint
import re

# Create a transformers inference pipeline
pipe = pipeline("text-generation", model=gemma_model, tokenizer=tokenizer)

user_prompt = "Schedule a \"team meeting\" tomorrow at 4pm."  #@param {type:"string"}
messages = [
    {"role": "developer", "content": "Current date and time given in YYYY-MM-DDTHH:MM:SS format: 2024-11-15T05:59:00. You are a model that can do function calling with the following functions"},
    {"role": "user", "content": user_prompt}
]

# Reuse the tools from the sample
tools = json.loads(dataset[0]['text'])['tools']

prompt = tokenizer.apply_chat_template(
    messages,
    tools=tools,
    tokenize=False,
    add_generation_prompt=True)

print(f"\n\033[1mPrompt:\033[0m {user_prompt}")
output = pipe(prompt, max_new_tokens=max_token_count)
model_output = output[0]['generated_text'][len(prompt):].strip()

print(f"\n\033[1mBase model output:\033[0m {model_output}")

Device set to use cuda:0



Prompt: Schedule a "team meeting" tomorrow at 4pm.

Base model output: I apologize, but I cannot assist with scheduling meetings. My current capabilities are limited to managing calendar events and contact management tools. I cannot generate or update meeting schedules.


請注意，基礎模型無法成功呼叫此prompt 的`create_calendar_event` 函數。
現在，我們將從訓練 dataset 中選取一個樣本，看看它的表現如何。

## 透過訓練進行測試dataset


In [ ]:
from transformers import pipeline
from random import randint
import re

# Create a transformers inference pipeline
pipe = pipeline("text-generation", model=gemma_model, tokenizer=tokenizer)

# Select a random sample from the test dataset
rand_idx = randint(0, len(train_dataset) - 1)
test_sample = train_dataset[rand_idx]

input_prompt = test_sample['prompt']
expected_output = test_sample['completion']

# Generate the output
output = pipe(input_prompt, max_new_tokens=max_token_count, skip_special_tokens=False)
actual_output = output[0]['generated_text'][len(input_prompt):].strip()

print(f"\n\033[1mInput prompt\033[0m   : {input_prompt}")
print(f"\n\033[1mExpected output\033[0m: {expected_output}")
print(f"\n\033[1mActual output\033[0m  : {actual_output}")

Device set to use cuda:0



Input prompt   : <bos><start_of_turn>developer
Current date and time given in YYYY-MM-DDTHH:MM:SS format: 2026-09-16T18:59:36
Day of week is Wednesday
You are a model that can do function calling with the following functions<start_function_declaration>declaration:turn_on_flashlight{description:<escape>Turns the flashlight on.<escape>,parameters:{type:<escape>OBJECT<escape>}}<end_function_declaration><start_function_declaration>declaration:create_contact{description:<escape>Creates a contact in the phone's contact list.<escape>,parameters:{properties:{email:{description:<escape>The email address of the contact.<escape>,type:<escape>STRING<escape>},first_name:{description:<escape>The first name of the contact.<escape>,type:<escape>STRING<escape>},last_name:{description:<escape>The last name of the contact.<escape>,type:<escape>STRING<escape>},phone_number:{description:<escape>The phone number of the contact.<escape>,type:<escape>STRING<escape>}},required:[<escape>first_name<escape>,<esc

基本模型輸出可能無法滿足您的期望 - 但這沒關係！
FunctionGemma 270M 是為任務專業化而設計的，這意味著當使用代表性範例進行訓練時，它可以提高特定任務的表現。讓我們微調模型以獲得更可靠的輸出。

## 微調模型

Hugging Face [TRL](https://huggingface.co/docs/trl/index) provides tools for training and fine-tuning LLMs.

### 設定調優作業
定義 FunctionGemma 基本模型的訓練設定。

In [ ]:
import torch
from transformers import AutoModelForCausalLM
from trl import SFTConfig

output_dir = "/content/mobile-actions-functiongemma"  # Where to save your fine-tuned checkpoints
tokenizer = AutoTokenizer.from_pretrained(gemma_model)

args = SFTConfig(
    output_dir=output_dir,                            # Directory to save adapters
    num_train_epochs=2,                               # Number of training epochs
    per_device_train_batch_size=4,                    # Batch size per device during training
    gradient_accumulation_steps=8,                    # Gradient accumulation during training
    logging_strategy="steps",                         # Log every steps
    eval_strategy="steps",                            # Evaluate loss metrics based on steps
    eval_steps=50,                                    # Evaluate loss metrics every 50 steps
    logging_steps=50,                                 # Log loss metrics every 50 steps
    save_strategy="epoch",                            # Save checkpoint every epoch
    learning_rate=1e-5,                               # Learning rate,
    lr_scheduler_type="cosine",                       # Cosine scheduler is often better for full FT
    max_length=max_token_count,                       # Max sequence length for model and packing of the dataset
    gradient_checkpointing=True,                      # Use gradient checkpointing to save memory
    packing=False,                                    # Groups multiple samples in the dataset into a single sequence
    optim="adamw_torch_fused",                        # Use fused adamw optimizer
    bf16=True,                                        # Use bf16 for mixed precision training
    completion_only_loss=True,                        # Train on completion only to improve quality
    report_to="none"                                  # No reporting.
)

base_model = AutoModelForCausalLM.from_pretrained(
    gemma_model,
    device_map="auto",
    dtype=torch.bfloat16,
    attn_implementation='eager')

base_model.config.pad_token_id = tokenizer.pad_token_id

print("Training configured")

Training configured


### 開始訓練

`SFTTrainer` token 化 dataset 並使用上一個步驟中的超參數訓練基本模型。
訓練時間會根據一系列因素而變化，例如 dataset 的大小或 epoch 的數量。使用 A100 GPU，1 個週期大約需要 8 分鐘。

In [ ]:
from trl import SFTTrainer

# Train and save the fine-tuned model
trainer = SFTTrainer(
    model=base_model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

trainer.train()

trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"Fine-tuned model saved to {output_dir}")

Adding EOS to train dataset:   0%|          | 0/8693 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/8693 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/8693 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/961 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/961 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/961 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2, 'pad_token_id': 0}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
50,0.034100,0.020567,0.534150,1049453.000000,0.994737
100,0.016000,0.017114,0.548209,2093954.000000,0.995557
150,0.015400,0.015405,0.543126,3134560.000000,0.996056
200,0.012900,0.014391,0.563650,4178739.000000,0.996431
250,0.010600,0.013738,0.553759,5224247.000000,0.996450
300,0.012300,0.013411,0.552985,6260126.000000,0.996673
350,0.008900,0.013343,0.537549,7306471.000000,0.996601
400,0.009200,0.013451,0.532500,8347062.000000,0.996600
450,0.009600,0.013342,0.532079,9390265.000000,0.996630
500,0.008800,0.013452,0.533780,10436648.000000,0.996691


Fine-tuned model saved to /content/mobile-actions-functiongemma


每次訓練checkpoint（時期）的權重將保存在您的臨時Colab會話儲存中。現在，您可以評估訓練和驗證損失指標，以選擇模型的checkpoint。

### 繪製訓練結果
要評估模型，您可以使用 Matplotlib 繪製訓練和驗證損失，以視覺化訓練步驟或曆元的這些指標。这有助于监控训练过程并就调整哪些超参数做出明智的决定。

In [ ]:
import matplotlib.pyplot as plt

# Access the log history
log_history = trainer.state.log_history

# Extract training / validation loss
train_losses = [log["loss"] for log in log_history if "loss" in log]
epoch_train = [log["epoch"] for log in log_history if "loss" in log]
eval_losses = [log["eval_loss"] for log in log_history if "eval_loss" in log]
epoch_eval = [log["epoch"] for log in log_history if "eval_loss" in log]

# Plot the training loss
plt.plot(epoch_train, train_losses, label="Training Loss")
plt.plot(epoch_eval, eval_losses, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss per Epoch")
plt.legend()
plt.grid(True)
plt.show()

### 測試微調後的模型

讓我們將微調後的模型性能與基本模型進行比較！透過更新 `user_prompt` 測試一些輸入。

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# Create Transformers inference pipeline
trained_model = AutoModelForCausalLM.from_pretrained(output_dir, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(output_dir)
pipe = pipeline("text-generation", model=trained_model, tokenizer=tokenizer)
pipe_base = pipeline("text-generation", model=gemma_model, device_map="auto")

# Test a prompt
user_prompt = "Schedule a \"team meeting\" tomorrow at 4pm."  #@param {type:"string"}
messages = [
    {"role": "developer", "content": "Current date and time given in YYYY-MM-DDTHH:MM:SS format: 2024-11-15T05:59:00. You are a model that can do function calling with the following functions"},
    {"role": "user", "content": user_prompt}
]

# Reuse the tools from the sample
tools = json.loads(dataset[0]['text'])['tools']

prompt = tokenizer.apply_chat_template(
    messages,
    tools=tools,
    tokenize=False,
    add_generation_prompt=True)

print(f"\n\033[1mPrompt:\033[0m {prompt}")
output = pipe(prompt, max_new_tokens=max_token_count)
output_base = pipe_base(prompt, max_new_tokens=max_token_count)
model_output = output[0]['generated_text'][len(prompt):].strip()
model_output_base = output_base[0]['generated_text'][len(prompt):].strip()

print(f"\n\033[1mFine-tuned model output:\033[0m {model_output}")

print(f"\n\033[1mBase model output:\033[0m       {model_output_base}")

### 評估微調模型

評估微調模型對於確保該過程確實提高了模型的性能而不引入新問題至關重要。

In [ ]:
#@title Helper functions for evaluation

import pandas as pd

def extract_function_call(model_output):
    """
    Parses a string containing specific function call markers and returns
    a list of function call objects. Here is an example of the obejct:

    ```
    <start_function_call>call:open_map{query:<escape>San Francisco<escape>}<end_function_call>
    ```

    Args:
        model_output (str): The model output string.

    Returns:
        list: A list of dictionaries representing the function calls.
    """
    results = []

    # Pattern to extract the full content of a single function call
    # Flags: DOTALL allows matching across newlines if necessary
    call_pattern = r"<start_function_call>(.*?)<end_function_call>"
    raw_calls = re.findall(call_pattern, model_output, re.DOTALL)

    for raw_call in raw_calls:
        # Check if the content starts with 'call:'
        if not raw_call.strip().startswith("call:"):
            continue

        # Extract function name
        # Expected format: call:func_name{...}
        try:
            # Split only on the first brace to separate name and args
            pre_brace, args_segment = raw_call.split("{", 1)

            function_name = pre_brace.replace("call:", "").strip()

            # Remove the trailing closing brace '}'
            args_content = args_segment.strip()
            if args_content.endswith("}"):
                args_content = args_content[:-1]

            arguments = {}

            # Pattern to extract arguments
            # Looks for: key:<escape>value<escape>
            # The key pattern [^:,]* ensures we don't accidentally eat previous commas
            arg_pattern = r"(?P<key>[^:,]*?):<escape>(?P<value>.*?)<escape>"

            arg_matches = re.finditer(arg_pattern, args_content, re.DOTALL)

            for match in arg_matches:
                key = match.group("key").strip()
                value = match.group("value")
                arguments[key] = value

            results.append({
                "function": {
                    "name": function_name,
                    "arguments": arguments
                }
            })

        except ValueError:
            # Handles cases where syntax might be malformed (e.g., missing '{')
            continue

    return results

def extract_text(model_output):
    """
    Extracts text content and removing the <end_of_turn> marker.

    Args:
        model_output (str): The model output string.

    Returns:
        str: The cleaned text.
    """
    if not model_output or model_output.startswith("<start_function_call>"):
        return None
    return model_output.replace("<end_of_turn>", "").strip()

from transformers import pipeline
from transformers.pipelines.pt_utils import KeyDataset

def get_eval_logs(dataset, pipe):
  batch_size = 1
  logs = []
  # Select a random sample from the test dataset
  for i, output in enumerate(pipe(KeyDataset(dataset, "prompt"), batch_size=batch_size)):
    orig_data = dataset[i]['text']
    messages = json.loads(orig_data)['messages']
    user_message = messages[1]
    assistant_first_message = messages[2]
    input_prompt = dataset[i]['prompt']
    # Generate the output
    model_output_only = output[0]['generated_text'][len(input_prompt):].strip()

    logs.append(
        {
            # The original user prompt/query.
            "user": user_message['content'],

            # List of ground truth function call objects.
            "target_fc": assistant_first_message.get('tool_calls', []),

            # Ground truth text response.
            "target_text": assistant_first_message.get('content'),

            # List of model-generated function call objects.
            "output_fc": extract_function_call(model_output_only),

            # Model-generated text response.
            "output_text": extract_text(model_output_only),
        }
    )

    if (i + 1) % batch_size == 0:
      print(f"Eval process: {(i + 1) * 100.0 / len(dataset):.2f}%")
  return logs

def get_scored_data_frame(dataset, pipe):
  logs = get_eval_logs(dataset, pipe)
  logs_df = pd.DataFrame.from_records(logs)

  scored = pd.DataFrame()
  scored['user'] = logs_df['user']
  scored['target_names'] = logs_df['target_fc'].apply(lambda x: [fc['function']['name'] for fc in x])
  scored['output_names'] = logs_df['output_fc'].apply(lambda x: [fc['function']['name'] for fc in x])
  scored["target_arguments"] = logs_df['target_fc'].apply(lambda x: [dict(sorted(fc['function']['arguments'].items())) for fc in x])
  scored["output_arguments"] = logs_df['output_fc'].apply(lambda x: [dict(sorted(fc['function']['arguments'].items())) for fc in x])
  scored['target_text'] = logs_df['target_text']
  scored['output_text'] = logs_df['output_text']
  scored["correct_names"] = scored["target_names"] == scored["output_names"]
  scored["correct_arguments"] = scored["target_arguments"] == scored["output_arguments"]
  scored["correct"] = scored["correct_names"] & scored["correct_arguments"]

  return scored

def review(scored):
  scored["incorrect_names"] = scored["target_names"] != scored["output_names"]
  scored["incorrect_arguments"] = scored["target_arguments"] != scored["output_arguments"]
  scored["incorrect"] = scored["incorrect_names"] | scored["incorrect_arguments"]

  for index, row in scored[scored["incorrect"]].iterrows():
    print(f"\033[1mSample #{index} prompt  \033[0m: {row["user"]}")
    print(f"\033[1mSample #{index} expected\033[0m: {row["target_names"]}, {row["target_arguments"]}")
    print(f"\033[1mSample #{index} actual  \033[0m: {row["output_names"]}, {row["output_arguments"]}")
    print("---------------")


In [ ]:
#@title Evaluate the base model

base_scored = get_scored_data_frame(
    eval_dataset,
    pipeline("text-generation", model=gemma_model, device_map="auto", temperature = 0.001),
)

base_scored

In [ ]:
#@title Evaluate the fine-tuned model

from transformers import pipeline
from random import randint
import re

# Create a transformers inference pipeline
trained_model = AutoModelForCausalLM.from_pretrained(output_dir, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(output_dir)

trained_scored = get_scored_data_frame(
    eval_dataset,
    pipeline("text-generation", model=trained_model, tokenizer=tokenizer, temperature = 0.001)
)

trained_scored

In [ ]:
#@title Compare the score of the base and fine-tuned models

# Optional: save the score in json file
trained_scored.to_json('scored_df_20251215_trained.json')
base_scored.to_json('scored_df_20251215_base.json')

print(f"\033[1mBase model score\033[0m       : {base_scored["correct"].mean()}")
print(f"\033[1mFine-tuned model score\033[0m : {trained_scored["correct"].mean()}")

## 查看不符的內容

分数告诉我们有多少 eval 数据得到了完全正确的函数调用。它給出了正確性的下限。某些输出可能与评估日期不匹配，但仍然可以接受。例如，下面的 `show_map` 函數呼叫：
* show_map:{'query': 'Maison Marulaz, 貝桑松, 法國'}
* show_map:{'query': '法國貝桑松 Maison Marulaz'}

讓我們來看看沒有完全匹配的評估日期。其中許多也是可以接受的。

In [ ]:
review(trained_scored)

該模型是否呼叫了您期望的函數？
如果您沒有得到想要的結果，您可以嘗試[使用不同的超參數](#scrollTo=-BJFoOdL0y8w) 來訓練模型，或更新您的訓練dataset 以包含更具代表性的範例。
對結果感到滿意後，您可以將模型儲存到Hugging Face Hub。

## 儲存您的模型並上傳至Hugging Face Hub
**您現在擁有了客製化的FunctionGemma 270M 型號！ 🎉**
將其上傳到 Hugging Face Hub 上的儲存庫，以便您輕鬆共享您的模型或稍後存取它。

In [ ]:
from huggingface_hub import ModelCard, ModelCardData, whoami

trained_model = AutoModelForCausalLM.from_pretrained(output_dir, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(output_dir)

#@markdown Name your model
model_name = "mobile-actions"    #@param {type:"string"}

username = whoami()['name']
hf_repo_id = f"{username}/functiongemma-270m-it-{model_name}"

repo_url = trained_model.push_to_hub(hf_repo_id, create_repo=True, commit_message="Upload model")
tokenizer.push_to_hub(hf_repo_id)

card_content = f"""
---
base_model: {gemma_model}
tags:
- function-calling
- mobile-actions
- gemma
---
A fine-tuned model based on `{gemma_model}`."""
card = ModelCard(card_content)

card.push_to_hub(hf_repo_id)

print(f"Uploaded to {repo_url}")

## 轉換為 .litertlm 以進行設備上部署

第一步是使用 `pip` 軟體包安裝程式安裝必要的庫。

In [ ]:
!pip uninstall -y tensorflow
!pip install ai-edge-torch-nightly --force-reinstall
!pip install ai-edge-litert-nightly

### 從微調模型建構 .litertlm

執行下面的腳本後，您可以在 colab 環境中的`/content/litertlm/mobile-actions_q8_ekv1024.litertlm`中找到轉換後的模型。將其複製到持久性儲存（例如 Google Drive）。

In [ ]:
import os
from ai_edge_torch.generative.examples.gemma3 import gemma3
from ai_edge_torch.generative.utilities import converter
from ai_edge_torch.generative.utilities.export_config import ExportConfig
from ai_edge_torch.generative.layers import kv_cache

# Metadata for FunctionGemma
llm_metadata = r"""start_token: {
    token_ids: {
        ids: [ 2 ]
    }
}
stop_tokens: {
    token_str: "<end_of_turn>"
}
stop_tokens: {
    token_str: "<start_function_response>"
}
llm_model_type: {
    function_gemma: {}
}
"""

checkpoint_dir = "/content/mobile-actions-functiongemma"

litertlm_output_dir = '/content/litertlm'
os.makedirs(litertlm_output_dir, exist_ok=True)

# Create the LLM metadata file
metadata_path = os.path.join(litertlm_output_dir, 'base_llm_metadata.textproto')
with open(metadata_path, 'w') as f:
  f.write(llm_metadata)

# Import the weights and build the PyTorch model
pytorch_model = gemma3.build_model_270m(checkpoint_dir)

# Setup the export configurations and parameters for text generation models.
export_config = ExportConfig()
export_config.kvcache_layout = kv_cache.KV_LAYOUT_TRANSPOSED
export_config.mask_as_input = True

# Convert to LiteRT-LM Format
converter.convert_to_litert(
    pytorch_model,
    output_path=litertlm_output_dir,
    output_name_prefix="mobile-actions",
    prefill_seq_len=256,
    kv_cache_max_len=1024,
    quantize="dynamic_int8",
    export_config=export_config,
    tokenizer_model_path=os.path.join(checkpoint_dir, 'tokenizer.model'),
    base_llm_metadata_path=metadata_path,
    output_format="litertlm",
)


### 將 `.litertlm` 儲存在 Google 雲端硬碟上

要將轉換後的模型部署到[Google AI Edge Gallery](https://play.google.com/store/apps/details?id=com.google.ai.edge.gallery)上，我們可以先將模型儲存在 Google Drive 上。在圖庫應用程式中，稍後從 Google 雲端硬碟匯入模型。

In [ ]:
#@title Mounting Google Drive on the colab environment

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#@title Save the `.litertlm` on Google Drive

!mkdir -p /content/drive/MyDrive/mobile-actions/
!cp /content/litertlm/mobile-actions_q8_ekv1024.litertlm /content/drive/MyDrive/mobile-actions/

## 摘要與後續步驟

恭喜！您已完成[針對移動操作微調FunctionGemma 270M](https://ai.google.dev/gemma/docs/mobile-actions)的第一部分。
您已成功使用 Mobile Actions dataset 微調 FunctionGemma 270M 並將其轉換為 `.litertlm` 格式。